In [1]:
library(plyr)
library(dplyr)
library(tidyr)
library(lme4)
library(lmerTest)
library(interactions)
library(ggplot2)
library(emmeans)
library(rempsyc)


Attaching package: 'dplyr'


The following objects are masked from 'package:plyr':

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack



Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

Suggested APA citation: Th<U+00E9>riault, R. (2023). rempsyc: Convenience functions for psychology. 
Journal of Open Source Software, 8(87), 5466. https://doi.org/10.21105/joss.05466



# TF measures predicted by acc * soc * age

## Load data

In [2]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'power_early',
    'ITPS_early',
    'ICPS_early_DLPFC_collapsed',
    'ICPS_early_MOTOR_collapsed',
    'ICPS_early_OCC_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_22_02_2026_23_31_39.csv"


Warning message:
"There was 1 warning in `filter()`.
i In argument: `!if_all(c(data_cols), is.na)`.
Caused by warning:
! Using an external vector in selections was deprecated in tidyselect 1.1.0.
i Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(data_cols)

  # Now:
  data %>% select(all_of(data_cols))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>."


## Power

In [16]:
label <- "power"
model <- lmer(power_early ~ acc * age_m * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
# lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: power_early ~ acc * age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1590.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.7967 -0.5339 -0.0085  0.5582  3.8812 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1479   0.3846  
 Residual             0.2633   0.5132  
Number of obs: 855, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)       0.015636   0.044918 226.874565   0.348 0.728087    
acc1             -0.735354   0.017600 623.311749 -41.780  < 2e-16 ***
age_m             0.167464   0.030969 228.434590   5.407 1.61e-07 ***
soc1              0.013577   0.017903 653.072684   0.758 0.448495    
sex1             -0.119092   0.030985 226.577448  -3.844 0.000158 ***
dp_inperson1     -0.004465   0.044967 226.122430  -0.099 0.920986    
ac

In [17]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.2559193,0.03584042,389.5641,0.1752747652,0.3365638,7.140521,9.179542e-12
2,Correct,0.0790088,0.03542712,378.9318,-0.0007144323,0.1587320,2.230179,2.632066e-02




acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.256       0.036   389.564      0.185       0.326       7.141     0.000
Correct          0.079       0.035   378.932      0.009       0.149       2.230     0.026

### Plot

In [56]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Power"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_561350890 
                   2

## ITPS

In [18]:
label <- "ITPS"
model <- lmer(ITPS_early ~ acc * age_m * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
# lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: ITPS_early ~ acc * age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 2349.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4948 -0.5754 -0.0713  0.4674  3.7266 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2141   0.4627  
 Residual             0.7261   0.8521  
Number of obs: 851, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.005852   0.061651 221.685804  -0.095 0.924460    
acc1             -0.184800   0.029284 617.888980  -6.311  5.3e-10 ***
age_m             0.146848   0.042514 222.799071   3.454 0.000661 ***
soc1              0.004613   0.029651 655.266865   0.156 0.876414    
sex1              0.087107   0.042478 220.413373   2.051 0.041486 *  
dp_inperson1      0.002897   0.061689 220.827605   0.047 0.962588    
acc

In [19]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.25582531,0.05190172,452.2180,0.13910400,0.3725466,4.9290332,2.323960e-06
2,Correct,0.03787141,0.05139765,445.4925,-0.07772214,0.1534650,0.7368314,4.616127e-01




acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.256       0.052   452.218      0.154       0.358       4.929     0.000
Correct          0.038       0.051   445.493     -0.063       0.139       0.737     0.462

### Plot

In [19]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "ITPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_1148129182 
                    2

## MFC-DLPFC ICPS

In [3]:
label <- "ICPS_DLPFC"
model <- lmer(ICPS_early_DLPFC_collapsed ~ acc * age_m * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
# lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: ICPS_early_DLPFC_collapsed ~ acc * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 2146.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.0190 -0.5262 -0.0515  0.4321  4.0769 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1712   0.4138  
 Residual             0.5821   0.7629  
Number of obs: 844, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.020438   0.055518 227.364591  -0.368 0.713121    
acc1             -0.455084   0.026339 614.763581 -17.278  < 2e-16 ***
age_m             0.204191   0.038211 226.439138   5.344 2.22e-07 ***
soc1             -0.016680   0.026662 650.943589  -0.626 0.531799    
sex1              0.023088   0.038110 222.411495   0.606 0.545254    
dp_inperson1      0.034419   0.055547 226.426991   

In [8]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.2987916,0.04667660,457.4166,0.193824943,0.4037582,6.401313,7.644664e-10
2,Correct,0.1095913,0.04630009,452.3003,0.005467496,0.2137151,2.366978,1.835419e-02


In [12]:
library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")



acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.299       0.047   457.417      0.207       0.391       6.401     0.000
Correct          0.110       0.046   452.300      0.019       0.201       2.367     0.018

### Plot

In [50]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Frontolateral ICPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14), 
        legend.position = "none", # Change the legend title font size# Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_118580619 
                   2

## MFC-MOTOR ICPS

In [13]:
label <- "ICPS_MOTOR"
model <- lmer(ICPS_early_MOTOR_collapsed ~ acc * age_m * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
# lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: ICPS_early_MOTOR_collapsed ~ acc * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1987

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.6635 -0.5330 -0.0267  0.4874  3.7932 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1558   0.3948  
 Residual             0.4687   0.6846  
Number of obs: 847, groups:  sub, 232

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)     -2.305e-02  5.116e-02  2.168e+02  -0.451    0.653    
acc1            -5.802e-01  2.360e-02  6.114e+02 -24.588  < 2e-16 ***
age_m            1.814e-01  3.534e-02  2.180e+02   5.133 6.30e-07 ***
soc1             1.306e-02  2.390e-02  6.457e+02   0.547    0.585    
sex1            -1.381e-02  3.530e-02  2.159e+02  -0.391    0.696    
dp_inperson1     4.824e-02  5.120e-02  2.160e+02   0.

In [15]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.28769250,0.04265601,431.2919,0.19174823,0.3836368,6.744477,9.905076e-11
2,Correct,0.07508435,0.04237804,425.7763,-0.02023902,0.1704077,1.771775,7.714709e-02




acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.288       0.043   431.292      0.204       0.372       6.744     0.000
Correct          0.075       0.042   425.776     -0.008       0.158       1.772     0.077

### Plot

In [48]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Midlateral ICPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),
    legend.position = "none", # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_909403265 
                   2

## MFC-OCC ICPS

In [54]:
label <- "ICPS_OCC"
model <- lmer(ICPS_early_OCC_collapsed ~ acc * age_m * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: ICPS_early_OCC_collapsed ~ acc * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 2210.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.7694 -0.5643  0.0088  0.5157  4.0000 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2195   0.4685  
 Residual             0.6217   0.7885  
Number of obs: 839, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)       0.018767   0.059943 218.351210   0.313   0.7545    
acc1             -0.398289   0.027345 609.370861 -14.565   <2e-16 ***
age_m             0.059511   0.041542 222.836668   1.433   0.1534    
soc1              0.004286   0.027692 641.611122   0.155   0.8770    
sex1             -0.068712   0.041450 219.287998  -1.658   0.0988 .  
dp_inperson1     -0.011619   0.059974 217.475762  -0.

Computing profile confidence intervals ...



In [62]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc | soc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "soc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "acc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed



,acc,soc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,NS,0.16583484,0.06474164,737.2565,0.02042576,0.31124393,2.5614867,0.02124066
2,Correct,NS,0.05331330,0.06297372,720.7926,-0.08813167,0.19475826,0.8465960,0.39750133
3,Error,S,-0.05990113,0.06300423,720.7878,-0.20141462,0.08161236,-0.9507479,0.34205107
4,Correct,S,0.07879570,0.06268348,716.4674,-0.06199913,0.21959054,1.2570410,0.34205107


,contrast,soc,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Correct - Error,NS,-0.1125215,0.07850276,610.3177,-0.2666899,0.04164678,-1.433345,0.15227133
2,Correct - Error,S,0.1386968,0.07686341,605.3476,-0.0122545,0.28964817,1.804458,0.07165635


,contrast,acc,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,S - NS,Error,-0.22573597,0.07915113,631.9718,-0.3811670,-0.07030494,-2.8519615,0.004487013
2,S - NS,Correct,0.02548241,0.07747581,626.0983,-0.1266615,0.17762632,0.3289079,0.742335310


### Plot

In [67]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Posterolateral ICPS"
x_label = "Age"
cond_labels = c("Non-social", "Social")

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              mod2="soc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              colors = c("red", "blue"),
              legend.main = "Accuracy",
              mod2.labels = cond_labels,
              plot.points = T) +
  theme_minimal() +
  theme(
    text = element_text(size = 20),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    legend.text = element_text(size = 14),                       # Change the legend text font size
    legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 20, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-1, 1, by = 1))
  # scale_linetype_manual(values = c("solid", "solid"))
dev.off()

agg_record_1006651352 
                    2

# DDM measures predicted by ICPS

## Load data

In [5]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'reversed_ratio_diff',
    'a_diff',
    'p_diff',
    'ter_diff',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_22_02_2026_23_31_39.csv"


## sda/rd (attentional ratio)

In [22]:
label <- "ratio_reversed"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 14 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m *  
    soc + ICPS_early_DLPFC_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1034

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.38629 -0.63160  0.05079  0.58236  2.54333 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1343   0.3665  
 Residual             0.8772   0.9366  
Number of obs: 350, groups:  sub, 209

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.036284   0.082481 197.208176
ICPS_early_OCC_diff_collapsed                0.156533   0.059939 317.098200
age_m                                        0.008302   0.058737 181.065772
soc1                                        -0.051132   0.051693 172.715766
ICPS_early

Computing profile confidence intervals ...



### Plot

In [23]:
png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = reversed_ratio_diff))
g + geom_smooth(method = "lm") + geom_point() + labs(y=plot_label, x="Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 77 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 77 rows containing missing values or values outside the scale range
(`geom_point()`)."


agg_record_140738080 
                   2

In [30]:
label <- "ratio_reversed_motor"
plot_label <- bquote(paste('Attentional Control', 'sd'['a'], '/r'['d'], '(Post-error - Post-correct)'))
model <- lmer(reversed_ratio_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m *  
    soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1070.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.43515 -0.60592  0.02835  0.57791  2.56940 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1570   0.3962  
 Residual             0.8622   0.9286  
Number of obs: 366, groups:  sub, 214

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                -5.879e-02  8.170e-02  2.085e+02
ICPS_early_MOTOR_diff_collapsed             4.220e-02  5.880e-02  3.179e+02
age_m                                      -2.669e-02  5.811e-02  1.983e+02
soc1                                       -5.748e-02  5.008e-02  1.860e+02
sex1                                       -3.194e-02  5.626

Computing profile confidence intervals ...



## a (boundary separation)

In [22]:
label <- "boundary_separation"
# plot_label <- bquote(atop("Boundary Separation", italic(ɑ) ~ "(Post-error - Post-correct)"))
model <- lmer(a_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 14 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
a_diff ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed *  
    age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1005.7

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74115 -0.66669 -0.04045  0.69570  2.56846 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.02404  0.1551  
 Residual             0.93066  0.9647  
Number of obs: 346, groups:  sub, 207

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.005964   0.078756 195.409592
ICPS_early_OCC_diff_collapsed                0.099667   0.057952 303.586318
age_m                                        0.105262   0.055069 175.075522
soc1                                         0.011053   0.052901 175.736139
ICPS_early_DLPFC_diff_coll

Computing profile confidence intervals ...



### Plot

In [40]:
png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
interact_plot(model, pred = "ICPS_early_DLPFC_diff_collapsed", modx = "age_m", interval = 1, dodge.width = 0.2,
              y.label = plot_label,
              x.label = "Midfrontal-Frontolateral ICPS (Error - Correct)",
              legend.main = "Age",
              plot.points = T) + 
  theme_minimal() +
  theme(
    text = element_text(size = 14), # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    # legend.text = element_text(size = 14),                       # Change the legend text font size
    # legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 18, face = "bold")         # Change the plot title font size and make it bold
  ) + theme(legend.position = "none") +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) + theme(legend.position = "bottom")
dev.off()

agg_record_481391806 
                   2

In [32]:
label <- "boundary separation_motor"
model <- lmer(a_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: a_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1056.4

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74175 -0.64152 -0.01014  0.64604  2.64944 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.04712  0.2171  
 Residual             0.95413  0.9768  
Number of obs: 362, groups:  sub, 212

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.01668    0.07946 203.51163
ICPS_early_MOTOR_diff_collapsed              0.02076    0.05770 295.71571
age_m                                        0.10671    0.05567 189.96732
soc1                                        -0.03466    0.05249 186.74996
sex1                                         0.12530    0.05393 187.23521
dp_inpers

Computing profile confidence intervals ...



# Behavioral post-error adjustments measures predicted by ICPS

## Load data

In [33]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'pea',
    'peri_rt',
    'pes',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
# tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_17_01_2026_19_46_00.csv"


## PEA

In [34]:
label <- "PEA"
model <- lmer(pea ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 14 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed *  
    age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1092.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5121 -0.5077  0.0688  0.6025  2.4833 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.09746  0.3122  
 Residual             0.84050  0.9168  
Number of obs: 379, groups:  sub, 218

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.078972   0.075825 200.302530
ICPS_early_OCC_diff_collapsed                0.105340   0.051838 348.954162
age_m                                        0.170152   0.053309 197.284888
soc1                                        -0.042320   0.048503 189.223596
ICPS_early_DLPFC_diff_collapsed        

Computing profile confidence intervals ...



In [35]:
label <- "PEA_motor"
model <- lmer(pea ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1166.2

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.2075 -0.5255  0.0365  0.6085  2.7822 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1598   0.3998  
 Residual             0.8044   0.8969  
Number of obs: 407, groups:  sub, 225

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.055345   0.075926 214.446803
ICPS_early_MOTOR_diff_collapsed              0.071772   0.051367 369.617960
age_m                                        0.162430   0.053000 218.871281
soc1                                        -0.035989   0.045719 208.137355
sex1                                         0.048018   0.052437 210.777741
dp_inpers

Computing profile confidence intervals ...



## PERI

In [36]:
label <- "PERI"
model <- lmer(peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 14 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed *  
    age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1093.1

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3584 -0.5425 -0.0293  0.5801  2.9487 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.08585  0.293   
 Residual             0.86682  0.931   
Number of obs: 377, groups:  sub, 217

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.029741   0.076707 215.526482
ICPS_early_OCC_diff_collapsed               -0.005724   0.052969 347.546032
age_m                                       -0.022499   0.053734 208.883185
soc1                                         0.075249   0.049376 201.550542
ICPS_early_DLPFC_diff_collapsed    

Computing profile confidence intervals ...



In [37]:
label <- "PERI_motor"
model <- lmer(peri_rt ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: peri_rt ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1181.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4106 -0.5572 -0.0146  0.5841  2.9674 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05914  0.2432  
 Residual             0.93866  0.9688  
Number of obs: 406, groups:  sub, 227

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.013553   0.074477 210.107070
ICPS_early_MOTOR_diff_collapsed             -0.044827   0.052063 347.766792
age_m                                       -0.013115   0.052148 215.715116
soc1                                         0.113377   0.049047 207.537956
sex1                                         0.001365   0.051171 206.090270
dp_inp

Computing profile confidence intervals ...



## PES

In [38]:
label <- "PES"
model <- lmer(pes ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 14 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed *  
    age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1068.3

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.97938 -0.54771  0.03617  0.56810  2.48590 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2665   0.5163  
 Residual             0.6409   0.8005  
Number of obs: 379, groups:  sub, 219

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.022388   0.079479 213.149404
ICPS_early_OCC_diff_collapsed               -0.022186   0.050763 364.016894
age_m                                        0.078792   0.055661 207.919529
soc1                                        -0.023202   0.043106 189.134214
ICPS_early_DLPFC_diff_collaps

Computing profile confidence intervals ...



In [39]:
label <- "PES_motor"
plot_y <- "PES"
model <- lmer(pes ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1142.3

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.99023 -0.54866  0.05665  0.56788  3.04109 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1796   0.4238  
 Residual             0.7332   0.8562  
Number of obs: 407, groups:  sub, 226

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.059613   0.074934 219.877195
ICPS_early_MOTOR_diff_collapsed              0.036512   0.050574 373.963814
age_m                                        0.025630   0.052093 221.697835
soc1                                        -0.051484   0.043741 209.703941
sex1                                         0.233921   0.051566 213.948463

Computing profile confidence intervals ...



# Updated DDM analyses

## Load data

In [2]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'reversed_ratio_diff',
    'a_diff',
    'p_diff',
    'ter_diff',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_01_02_2026_18_50_11.csv"


Warning message:
"There was 1 warning in `filter()`.
i In argument: `!if_all(c(data_cols), is.na)`.
Caused by warning:
! Using an external vector in selections was deprecated in tidyselect 1.1.0.
i Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(data_cols)

  # Now:
  data %>% select(all_of(data_cols))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>."


## sda/rd (attentional ratio)

In [5]:
label <- "ratio_reversed_frontolateral"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(reversed_ratio_diff ~ ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_DLPFC_diff_collapsed * age_m *  
    soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1060.7

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.54301 -0.56681  0.05757  0.53203  2.53325 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1728   0.4157  
 Residual             0.8702   0.9328  
Number of obs: 360, groups:  sub, 213

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.043459   0.084253 212.771194
ICPS_early_DLPFC_diff_collapsed              0.043162   0.060057 347.310941
age_m                                       -0.007254   0.059568 193.850229
soc1                                        -0.066001   0.050510 179.668538
sex1                                        -0.043750   0.05

Computing profile confidence intervals ...



In [6]:
label <- "ratio_reversed_midlateral"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(reversed_ratio_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m *  
    soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1070.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.43515 -0.60592  0.02835  0.57791  2.56940 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1570   0.3962  
 Residual             0.8622   0.9286  
Number of obs: 366, groups:  sub, 214

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                -5.879e-02  8.170e-02  2.085e+02
ICPS_early_MOTOR_diff_collapsed             4.220e-02  5.880e-02  3.179e+02
age_m                                      -2.669e-02  5.811e-02  1.983e+02
soc1                                       -5.748e-02  5.008e-02  1.860e+02
sex1                                       -3.194e-02  5.626

Computing profile confidence intervals ...



In [7]:
label <- "ratio_reversed_posterolateral"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m *  
    soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1050.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.43674 -0.63031  0.04997  0.58130  2.58214 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1392   0.3731  
 Residual             0.8309   0.9115  
Number of obs: 365, groups:  sub, 213

Fixed effects:
                                           Estimate Std. Error         df
(Intercept)                               -0.029941   0.077988 194.168513
ICPS_early_OCC_diff_collapsed              0.166999   0.056442 326.085882
age_m                                      0.016512   0.055784 183.984510
soc1                                      -0.049044   0.048682 176.659494
sex1                                      -0.027530   0.054754 184.91680

Computing profile confidence intervals ...



In [3]:
label <- "ratio_reversed_all_three"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 18 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m *  
    soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed *  
    age_m * soc + sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1026.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.3709 -0.6152  0.0267  0.5501  2.5955 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1092   0.3304  
 Residual             0.8995   0.9484  
Number of obs: 344, groups:  sub, 207

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.064240   0.083338 193.636781
ICPS_early_OCC_diff_collapsed                0.167163   0.061823 309.840092
age_m                                       -0.011185   0.059845 177.328462
soc1                                        -0

Computing profile confidence intervals ...



## a (boundary separation)

In [31]:
label <- "boundary_frontolateral"
plot_label <- 'boundary separation'
model <- lmer(a_diff ~ ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_DLPFC_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1149.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.14453 -0.52595  0.03846  0.59730  2.74918 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1465   0.3827  
 Residual             0.8299   0.9110  
Number of obs: 399, groups:  sub, 224

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.07476    0.07811 216.77836
ICPS_early_DLPFC_diff_collapsed              0.02406    0.05169 386.09042
age_m                                        0.18166    0.05354 204.88817
soc1                                        -0.02942    0.04671 195.26470
sex1                                         0.03842    0.05266 197.53340
dp_inperson

Computing profile confidence intervals ...



In [27]:
png(file=sprintf("%s/%s_stats_frontolateral_model_only.png", pic_path, label), width=9, height=5, units="in", res=600)
interact_plot(model, pred = "ICPS_early_DLPFC_diff_collapsed", modx = "age_m", interval = 1, dodge.width = 0.2,
              y.label = plot_label,
              x.label = "Midfrontal-Frontolateral ICPS (Error - Correct)",
              legend.main = "Age",
              plot.points = T) + 
  theme_minimal() +
  theme(
    text = element_text(size = 14), # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    # legend.text = element_text(size = 14),                       # Change the legend text font size
    # legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 18, face = "bold")         # Change the plot title font size and make it bold
  ) + theme(legend.position = "none") +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) + theme(legend.position = "bottom")
dev.off()

agg_record_1088077279 
                    2

In [9]:
label <- "boundary_midlateral"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(a_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: a_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1056.4

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74175 -0.64152 -0.01014  0.64604  2.64944 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.04712  0.2171  
 Residual             0.95413  0.9768  
Number of obs: 362, groups:  sub, 212

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.01668    0.07946 203.51163
ICPS_early_MOTOR_diff_collapsed              0.02076    0.05770 295.71571
age_m                                        0.10671    0.05567 189.96732
soc1                                        -0.03466    0.05249 186.74996
sex1                                         0.12530    0.05393 187.23521
dp_inpers

Computing profile confidence intervals ...



In [10]:
label <- "boundary_posterolateral"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))
model <- lmer(a_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: a_diff ~ ICPS_early_OCC_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1042.4

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74633 -0.64997 -0.03642  0.64555  2.59190 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.02805  0.1675  
 Residual             0.94136  0.9702  
Number of obs: 361, groups:  sub, 211

Fixed effects:
                                           Estimate Std. Error         df
(Intercept)                               -0.037514   0.076146 191.084811
ICPS_early_OCC_diff_collapsed              0.071889   0.055943 306.949489
age_m                                      0.103927   0.053573 177.231937
soc1                                      -0.004811   0.051572 179.445257
sex1                                       0.144133   0.052680 180.074088
dp_inperson

Computing profile confidence intervals ...



In [4]:
label <- "boundary_all_three"
plot_label <- 'boundary separation'
model <- lmer(a_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 18 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
a_diff ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_MOTOR_diff_collapsed *  
    age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m * soc +  
    sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1003.4

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.77903 -0.63181 -0.04632  0.66306  2.49593 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05292  0.2300  
 Residual             0.91714  0.9577  
Number of obs: 340, groups:  sub, 205

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.025425   0.081776 190.971682
ICPS_early_OCC_diff_collapsed                0.106733   0.060734 302.728945
age_m                                        0.110524   0.057779 171.009499
soc1                                         0.0

Computing profile confidence intervals ...



In [29]:
png(file=sprintf("%s/%s_stats_all_three.png", pic_path, label), width=9, height=5, units="in", res=600)
interact_plot(model, pred = "ICPS_early_DLPFC_diff_collapsed", modx = "age_m", interval = 1, dodge.width = 0.2,
              y.label = plot_label,
              x.label = "Midfrontal-Frontolateral ICPS (Error - Correct)",
              legend.main = "Age",
              plot.points = T) + 
  theme_minimal() +
  theme(
    text = element_text(size = 14), # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    # legend.text = element_text(size = 14),                       # Change the legend text font size
    # legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 18, face = "bold")         # Change the plot title font size and make it bold
  ) + theme(legend.position = "none") +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) + theme(legend.position = "bottom")
dev.off()

agg_record_52148715 
                  2

In [5]:
library(emmeans)

# 1. Directly specify the standardized values (-1 SD, Mean, +1 SD)
age_points <- list(age_m = c(-1, 0, 1))

# 2. Compute the simple slopes
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 3. View the slopes with 95% Confidence Intervals and p-values
summary(slopes, infer = c(TRUE, TRUE))

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,age_m,ICPS_early_DLPFC_diff_collapsed.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,-1,0.17390826,0.11321865,321.9372,-0.0488336,0.39665011,1.5360389,0.1255110
2,0,0.02348432,0.07126169,320.1717,-0.1167160,0.16368463,0.3295504,0.7419552
3,1,-0.12693962,0.09251097,316.7672,-0.3089532,0.05507398,-1.3721574,0.1709853


In [6]:
library(emmeans)

# 1. Compute the slopes at the standardized age points
age_points <- list(age_m = c(-1, 0, 1))
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 2. Compute paired comparisons between the slopes
pairwise_diffs <- pairs(slopes)

# 3. View the results
summary(pairwise_diffs, infer = c(TRUE, TRUE))

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,contrast,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,(age_m-1) - age_m0,0.1504239,0.07490089,321.3321,-0.02593965,0.3267875,2.008306,0.1118253
2,(age_m-1) - age_m1,0.3008479,0.14980179,321.3321,-0.05187930,0.6535751,2.008306,0.1118253
3,age_m0 - age_m1,0.1504239,0.07490089,321.3321,-0.02593965,0.3267875,2.008306,0.1118253


# Updated raw behav measures

## Load data

In [30]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'pea',
    'peri_rt',
    'pes',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
# tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_01_02_2026_18_50_11.csv"


## PEA

In [33]:
label <- "pea_frontolateral"
plot_label <- 'PEA'
model <- lmer(pea ~ ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_DLPFC_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1149.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.14453 -0.52595  0.03846  0.59730  2.74918 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1465   0.3827  
 Residual             0.8299   0.9110  
Number of obs: 399, groups:  sub, 224

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.07476    0.07811 216.77836
ICPS_early_DLPFC_diff_collapsed              0.02406    0.05169 386.09042
age_m                                        0.18166    0.05354 204.88817
soc1                                        -0.02942    0.04671 195.26470
sex1                                         0.03842    0.05266 197.53340
dp_inperson

Computing profile confidence intervals ...



In [34]:
label <- "pea_midlateral"
plot_label <- 'PEA'
model <- lmer(pea ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1166.2

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.2075 -0.5255  0.0365  0.6085  2.7822 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1598   0.3998  
 Residual             0.8044   0.8969  
Number of obs: 407, groups:  sub, 225

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.055345   0.075926 214.446803
ICPS_early_MOTOR_diff_collapsed              0.071772   0.051367 369.617960
age_m                                        0.162430   0.053000 218.871281
soc1                                        -0.035989   0.045719 208.137355
sex1                                         0.048018   0.052437 210.777741
dp_inpers

Computing profile confidence intervals ...



In [35]:
label <- "pea_posterolateral"
plot_label <- 'PEA'
model <- lmer(pea ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_OCC_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1127

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5958 -0.5665  0.0732  0.6211  2.4881 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1172   0.3424  
 Residual             0.8076   0.8987  
Number of obs: 398, groups:  sub, 222

Fixed effects:
                                          Estimate Std. Error        df t value
(Intercept)                               -0.06954    0.07240 200.92345  -0.960
ICPS_early_OCC_diff_collapsed              0.10114    0.04969 368.11030   2.036
age_m                                      0.16606    0.05137 210.73889   3.233
soc1                                      -0.05072    0.04593 199.79856  -1.104
sex1                                       0.05216    0.05099 203.332

Computing profile confidence intervals ...



In [36]:
label <- "pea_all_three"
plot_label <- 'PEA'
model <- lmer(pea ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 18 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pea ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_MOTOR_diff_collapsed *  
    age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m * soc +  
    sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1083.9

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5095 -0.5179  0.0762  0.5733  2.4774 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1029   0.3208  
 Residual             0.8351   0.9138  
Number of obs: 372, groups:  sub, 216

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.083351   0.077675 199.710114
ICPS_early_OCC_diff_collapsed                0.108243   0.054289 345.715197
age_m                                        0.162057   0.054710 196.318137
soc1                                        -0.035146   0.049

Computing profile confidence intervals ...



## PERI

In [37]:
label <- "peri_frontolateral"
plot_label <- 'PERI'
model <- lmer(peri_rt ~ ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: peri_rt ~ ICPS_early_DLPFC_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1161.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3451 -0.5208 -0.0038  0.5702  2.9370 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.09447  0.3074  
 Residual             0.91459  0.9563  
Number of obs: 398, groups:  sub, 225

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.013601   0.078577 226.357971
ICPS_early_DLPFC_diff_collapsed              0.030113   0.053158 380.750425
age_m                                       -0.031146   0.053579 211.831720
soc1                                         0.105635   0.049020 204.663171
sex1                                        -0.006591   0.052418 203.226486
dp_inp

Computing profile confidence intervals ...



In [38]:
label <- "peri_midlateral"
plot_label <- 'PERI'
model <- lmer(peri_rt ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: peri_rt ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1181.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4106 -0.5572 -0.0146  0.5841  2.9674 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05914  0.2432  
 Residual             0.93866  0.9688  
Number of obs: 406, groups:  sub, 227

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.013553   0.074477 210.107070
ICPS_early_MOTOR_diff_collapsed             -0.044827   0.052063 347.766792
age_m                                       -0.013115   0.052148 215.715116
soc1                                         0.113377   0.049047 207.537956
sex1                                         0.001365   0.051171 206.090270
dp_inp

Computing profile confidence intervals ...



In [39]:
label <- "peri_posterolateral"
plot_label <- 'PERI'
model <- lmer(peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m * soc + sex +  
    dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1128.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3481 -0.5497 -0.0028  0.5615  2.9882 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.08612  0.2935  
 Residual             0.85320  0.9237  
Number of obs: 396, groups:  sub, 221

Fixed effects:
                                          Estimate Std. Error        df t value
(Intercept)                                0.04466    0.07239 202.08298   0.617
ICPS_early_OCC_diff_collapsed             -0.01042    0.05074 360.91437  -0.205
age_m                                     -0.01460    0.05131 208.66086  -0.284
soc1                                       0.07498    0.04716 199.63549   1.590
sex1                                      -0.01590    0.05084 20

Computing profile confidence intervals ...



In [40]:
label <- "peri_all_three"
plot_label <- 'PERI'
model <- lmer(peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 18 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_MOTOR_diff_collapsed *  
    age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m * soc +  
    sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1088.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3334 -0.5924 -0.0033  0.5408  2.9411 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.08003  0.2829  
 Residual             0.87657  0.9363  
Number of obs: 371, groups:  sub, 215

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 2.611e-02  7.820e-02  2.131e+02
ICPS_early_OCC_diff_collapsed               1.290e-02  5.498e-02  3.431e+02
age_m                                      -1.297e-03  5.505e-02  2.074e+02
soc1                                        9.307e-02  5.

Computing profile confidence intervals ...



## PES

In [46]:
label <- "pes_frontolateral"
plot_label <- 'PES'
model <- lmer(pes ~ ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_DLPFC_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1116.3

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.07475 -0.56359  0.02076  0.57474  2.61654 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2676   0.5173  
 Residual             0.6553   0.8095  
Number of obs: 399, groups:  sub, 224

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                -2.140e-02  7.960e-02  2.293e+02
ICPS_early_DLPFC_diff_collapsed            -8.071e-02  4.990e-02  3.882e+02
age_m                                       8.339e-02  5.450e-02  2.153e+02
soc1                                       -3.154e-02  4.186e-02  1.980e+02
sex1                                        2.361e-01  5.380e-02  2.081e+02

Computing profile confidence intervals ...



In [47]:
label <- "pes_midlateral"
plot_label <- 'PES'
model <- lmer(pes ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_MOTOR_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1142.3

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.99023 -0.54866  0.05665  0.56788  3.04109 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1796   0.4238  
 Residual             0.7332   0.8562  
Number of obs: 407, groups:  sub, 226

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.059613   0.074934 219.877195
ICPS_early_MOTOR_diff_collapsed              0.036512   0.050574 373.963814
age_m                                        0.025630   0.052093 221.697835
soc1                                        -0.051484   0.043741 209.703941
sex1                                         0.233921   0.051566 213.948463

Computing profile confidence intervals ...



In [48]:
label <- "pes_posterolateral"
plot_label <- 'PES'
model <- lmer(pes ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_OCC_diff_collapsed * age_m * soc + sex + dp_inperson +  
    (1 | sub)
   Data: tf_data

REML criterion at convergence: 1106.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.03988 -0.56122  0.03695  0.58383  2.59889 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2191   0.4681  
 Residual             0.6762   0.8223  
Number of obs: 398, groups:  sub, 223

Fixed effects:
                                          Estimate Std. Error        df t value
(Intercept)                               -0.04212    0.07474 207.53236  -0.564
ICPS_early_OCC_diff_collapsed             -0.02870    0.04894 381.52798  -0.587
age_m                                      0.04788    0.05261 214.14821   0.910
soc1                                      -0.03626    0.04236 197.52635  -0.856
sex1                                       0.23861    0.0

Computing profile confidence intervals ...



In [49]:
label <- "pes_all_three"
plot_label <- 'PES'
model <- lmer(pes ~ ICPS_early_OCC_diff_collapsed * age_m  * soc + ICPS_early_MOTOR_diff_collapsed * age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m  * soc + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))


Correlation matrix not shown by default, as p = 18 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: 
pes ~ ICPS_early_OCC_diff_collapsed * age_m * soc + ICPS_early_MOTOR_diff_collapsed *  
    age_m * soc + ICPS_early_DLPFC_diff_collapsed * age_m * soc +  
    sex + dp_inperson + (1 | sub)
   Data: tf_data

REML criterion at convergence: 1060.8

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.79992 -0.56302  0.03411  0.57897  2.67977 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2336   0.4833  
 Residual             0.6668   0.8166  
Number of obs: 372, groups:  sub, 217

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.01850    0.07984 212.63271
ICPS_early_OCC_diff_collapsed               -0.03491    0.05295 353.80655
age_m                                        0.04917    0.05615 207.74398
soc1                                        -0.04070    0.0

Computing profile confidence intervals ...

